# Memorization Type 2: Home–Work Association (same home, different work)

This notebook is part of the broader memorization analysis framework for mobility sequence predictors. The focus here is on evaluating **memorization of home–work location associations**: how much the model relies on memorized links between a user's **home** and **work** locations.

### Memorization Type 2: What we test

We want to evaluate whether the model memorizes that a user who lives in a specific location **always** goes to a specific work location — and whether it generalizes this behavior appropriately across different users who share a home location.

In other words, we compare:

- **Training trajectories** where a user moves between a consistent home–work pair.
- **Reference trajectories** from other users who **share the same home location** but **have different work locations**.


### This notebook performs the following:

1. **Load and preprocess** the user trajectory dataset.
2. **Infer home and work locations** per user based on time-of-day patterns.
3. **Group users by shared home location**.
4. Within each group:
   - Select one user’s trajectory as the training sample.
   - Use the other users (with different work locations) as the **reference set**.
5. **Evaluate** the trained model on both training and reference sets.
6. **Quantify memorization** via likelihood-based metrics (e.g., perplexity, rank, likelihood gap).


### Objective

This setup allows us to evaluate:
> Does the model memorize the specific *home → work* link, or does it generalize across users with the same home but different work habits?


In [17]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from haversine import haversine
import matplotlib.pyplot as plt
import random
sys.path.append('./Helpers/')
from utils import save_dict, load_dict
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import pairwise_distances
from collections import defaultdict
from sklearn.cluster import KMeans
from fastdtw import fastdtw
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
import numpy as np
from tqdm import tqdm  
from geopy.distance import geodesic

In [57]:
DATASET_NAME_TO_FOLDER_NAME = {
    'boston': 'Boston',
    'geolife': 'Geolife',
    'shenzhenurban': 'ShenzhenUrban',
    'shanghaikaggle': 'ShanghaiKaggle',
    'yjmob100k': 'YJMob100Kv3'
}
dataset_name = "yjmob100k"
version = 0

save_path_base_home = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Home'
)

save_path_base_work = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Work'
)


save_path_data_abstraction_home = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Home' /
    'abstracted_trajectory_records.dict'
)

save_path_data_abstraction_work = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Work' /
    'abstracted_trajectory_records.dict'
)



save_path_data_datasets_home = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Home' /
    'Datasets/'
)

save_path_data_datasets_work = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType2' / 'Work' /
    'Datasets/'
)

In [19]:
df = pd.read_pickle(Path('PreprocessedData/1FilteredData/') / DATASET_NAME_TO_FOLDER_NAME[dataset_name]/str(version)/'dataset.pkl', compression=None)
location_df = pd.read_csv(Path('PreprocessedData/1FilteredData/') / DATASET_NAME_TO_FOLDER_NAME[dataset_name]/str(version)/'dictionary.csv')
location_dict = location_df.set_index('Location')[['Latitude', 'Longitude']].apply(tuple, axis=1).to_dict()

In [20]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['date'] = df['Timestamp'].dt.date
df['Latitude'] = df['Location'].map(location_dict).str[0]
df['Longitude'] = df['Location'].map(location_dict).str[1]
traj_df = df[['DeviceID', 'date']].drop_duplicates().reset_index().rename(columns={'index': 'traj_id'})
df = df.merge(right=traj_df, on=['DeviceID', 'date'])

## Dataset abstraction

In [25]:
def haversine_vectorized(origin, destinations):
    R = 6371  # Earth radius in kilometers
    lat1 = np.radians(origin[0])
    lon1 = np.radians(origin[1])
    lat2 = np.radians(destinations[:, 0])
    lon2 = np.radians(destinations[:, 1])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c *1000  # distance in m


def abstract_k_day_by_home(df, k=3):
    """
    Abstract k-day trajectories by [home_lat, home_lon, distance_home_to_work].
    """

    trajectory_records = []
    expected_len = 48 * k

    for device_id, user_df in df.groupby('DeviceID'):
        user_df = user_df.sort_values('Timestamp')
        unique_trajs = sorted(user_df['traj_id'].unique())

        for i in range(0, len(unique_trajs) - k + 1, k):
            selected_trajs = unique_trajs[i:i + k]
            chunk_df = user_df[user_df['traj_id'].isin(selected_trajs)].sort_values('Timestamp')

            if len(chunk_df) != expected_len:
                continue

            chunk_df['hour'] = pd.to_datetime(chunk_df['Timestamp']).dt.hour
            home_df = chunk_df[chunk_df['hour'] < 6]
            work_df = chunk_df[(chunk_df['hour'] >= 9) & (chunk_df['hour'] < 17)]

            if home_df.empty or work_df.empty:
                continue

            home_loc = home_df['Location'].mode()
            work_loc = work_df['Location'].mode()
            if home_loc.empty or work_loc.empty:
                continue

            home_coords = home_df[home_df['Location'] == home_loc.iloc[0]][['Latitude', 'Longitude']].dropna()
            work_coords = work_df[work_df['Location'] == work_loc.iloc[0]][['Latitude', 'Longitude']].dropna()

            if home_coords.empty or work_coords.empty:
                continue

            home_center = tuple(home_coords.mean())
            work_center = tuple(work_coords.mean())
            if dataset_name == "yjmob100k":
                distance_m = np.linalg.norm(np.array(home_center) - np.array(work_center))
            else:
                distance_m = haversine(home_center, work_center)*1000

            pattern = [home_center[0], home_center[1], distance_m]

            trajectory_records.append({
                'traj_id': f"{selected_trajs[0]}_{selected_trajs[-1]}",
                'home':home_center,
                'work':work_center,
                'DeviceID': device_id,
                'pattern': pattern  
            })

    return trajectory_records



def abstract_k_day_by_work(df, k=3):
    """
    Abstract k-day trajectories by [work_lat, work_lon, distance_home_to_work].
    """

    trajectory_records = []
    expected_len = 48 * k

    for device_id, user_df in df.groupby('DeviceID'):
        user_df = user_df.sort_values('Timestamp')
        unique_trajs = sorted(user_df['traj_id'].unique())

        for i in range(0, len(unique_trajs) - k + 1, k):
            selected_trajs = unique_trajs[i:i + k]
            chunk_df = user_df[user_df['traj_id'].isin(selected_trajs)].sort_values('Timestamp')

            if len(chunk_df) != expected_len:
                continue

            chunk_df['hour'] = pd.to_datetime(chunk_df['Timestamp']).dt.hour
            home_df = chunk_df[chunk_df['hour'] < 6]
            work_df = chunk_df[(chunk_df['hour'] >= 9) & (chunk_df['hour'] < 17)]

            if home_df.empty or work_df.empty:
                continue

            home_loc = home_df['Location'].mode()
            work_loc = work_df['Location'].mode()
            if home_loc.empty or work_loc.empty:
                continue

            home_coords = home_df[home_df['Location'] == home_loc.iloc[0]][['Latitude', 'Longitude']].dropna()
            work_coords = work_df[work_df['Location'] == work_loc.iloc[0]][['Latitude', 'Longitude']].dropna()

            if home_coords.empty or work_coords.empty:
                continue

            home_center = tuple(home_coords.mean())
            work_center = tuple(work_coords.mean())
            if dataset_name == "yjmob100k":
                distance_m = np.linalg.norm(np.array(home_center) - np.array(work_center))
            else:
                distance_m = haversine(home_center, work_center)*1000

            pattern = [work_center[0], work_center[1], distance_m]

            trajectory_records.append({
                'traj_id': f"{selected_trajs[0]}_{selected_trajs[-1]}",
                'home':home_center,
                'work':work_center,
                'DeviceID': device_id,
                'pattern': pattern  
            })

    return trajectory_records

In [27]:
# traj_id_list = list(df.traj_id.unique())
# df_short = df[df['traj_id'].isin(traj_id_list[14:50])]
# abstract_k_day_by_work(df_short)

In [28]:
trajectory_records_home_k3 = abstract_k_day_by_home(df)

In [29]:
#Saving the first part after normalization
save_dict(trajectory_records_home_k3, save_path_data_abstraction_home)

Saved to: PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Home/abstracted_trajectory_records.dict


In [30]:
trajectory_records_work_k3 = abstract_k_day_by_work(df)

In [ ]:
save_dict(trajectory_records_work_k3, save_path_data_abstraction_work)

Saved to: PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Work/abstracted_trajectory_records.dict


## Trajectories Clustering

In [42]:
current_selection = 'Work'
if current_selection == 'Home':
    trajectory_records = load_dict(save_path_data_abstraction_home)
elif current_selection == 'Work':
    trajectory_records = load_dict(save_path_data_abstraction_work)

Loaded from: PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Work/abstracted_trajectory_records.dict


In [43]:
nclusters = 2000
min_cluster_size = 100

In [44]:
X = [np.array(traj['pattern']).flatten() for traj in trajectory_records]
X = np.stack(X)

# Run KMeans
kmeans = KMeans(n_clusters=nclusters, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X)

# Assign cluster labels back
for traj, label in zip(trajectory_records, labels):
    traj['cluster'] = label

In [45]:
def get_cluster_medoids_safe(
    trajectory_records,
    kmeans_centroids,
    max_cluster_size=500,
    epsilon_init=0.05,
    epsilon_min=1e-4,
    epsilon_max=0.5,
    max_iterations=10
):
    """
    For small clusters (≤ max_cluster_size): compute true medoid (pairwise).
    For large clusters: adaptively search for bounding box around centroid to select points,
    then pick closest one to the centroid.

    Returns:
        List of medoid trajectory dicts (1 per cluster)
    """
    cluster_groups = defaultdict(list)
    for traj in trajectory_records:
        cluster_groups[traj['cluster']].append(traj)

    medoids = []

    for cluster_id, trajs in cluster_groups.items():
        centroid = kmeans_centroids[cluster_id]
        centroid = centroid / (np.linalg.norm(centroid) + 1e-10)

        vectors = [np.array(t['pattern']).flatten() for t in trajs]
        vectors = np.stack(vectors)
        vectors = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-10)

        if len(trajs) <= max_cluster_size:
            # Safe to compute true medoid
            dists = pairwise_distances(vectors, metric='euclidean')
            medoid_idx = np.argmin(dists.sum(axis=1))
            medoid_traj = trajs[medoid_idx]
        else:
            # Adaptive bounding box
            epsilon = epsilon_init
            best_filtered = None

            for _ in range(max_iterations):
                lower = centroid - epsilon
                upper = centroid + epsilon
                mask = np.all((vectors >= lower) & (vectors <= upper), axis=1)
                filtered = np.where(mask)[0]

                if len(filtered) == 0:
                    epsilon *= 2  # too narrow, expand
                    if epsilon > epsilon_max:
                        break
                elif len(filtered) > max_cluster_size:
                    epsilon /= 2  # too wide, shrink
                    if epsilon < epsilon_min:
                        break
                else:
                    best_filtered = filtered
                    break  # good range

            # If we found a filtered set, use it
            if best_filtered is not None and len(best_filtered) > 0:
                filtered_vecs = vectors[best_filtered]
                dists = pairwise_distances([centroid], filtered_vecs, metric='euclidean')[0]
                closest_idx = best_filtered[np.argmin(dists)]
                medoid_traj = trajs[closest_idx]
            else:
                # Fallback: use closest to centroid from all
                dists = pairwise_distances([centroid], vectors, metric='euclidean')[0]
                medoid_idx = np.argmin(dists)
                medoid_traj = trajs[medoid_idx]

        medoid_traj['cluster'] = cluster_id
        medoids.append(medoid_traj)

    return medoids

medoids = get_cluster_medoids_safe(
    trajectory_records,
    kmeans_centroids=kmeans.cluster_centers_,
    max_cluster_size=500
)

In [46]:
# Group trajectories by cluster
cluster_groups = defaultdict(list)
for traj in trajectory_records:
    cluster_id = traj['cluster']
    cluster_groups[cluster_id].append(traj)

#cluster sizes
cluster_sizes = {"key":[], "size":[]}
for cluster in cluster_groups:
   cluster_sizes['key'].append(cluster)
   cluster_sizes['size'].append(len(cluster_groups[cluster]))
cluster_sizes = pd.DataFrame(cluster_sizes)
cluster_sizes.describe()

,key,size
count,2000.000000,2000.000000
mean,999.500000,198.864500
std,577.494589,165.228903
min,0.000000,3.000000
25%,499.750000,76.000000
50%,999.500000,152.500000
75%,1499.250000,271.000000
max,1999.000000,1107.000000


In [47]:

def build_reference_sets_cluster_enrichment(
    trajectory_records,
    medoids,  # List of medoid traj dicts, 1 per cluster
    min_cluster_size=100,
):
    """
    Enrich small clusters with points from nearby clusters, using medoid-to-medoid distances.
    
    Parameters:
        trajectory_records: list of all trajectory dicts (with 'traj_id', 'cluster', 'pattern')
        medoids: list of cluster medoid trajectory dicts (1 per cluster)
        min_cluster_size: minimum number of references desired

    Returns:
        training_set: list of training trajectories (medoids)
        reference_sets: dict {train_traj_id: list of reference trajectories}
    """
    # Build lookup tables
    cluster_groups = defaultdict(list)
    for traj in trajectory_records:
        cluster_groups[traj['cluster']].append(traj)

    all_clusters = sorted(cluster_groups.keys())
    medoid_vectors = [np.array(m['pattern']).flatten() for m in medoids]
    medoid_vectors = np.stack(medoid_vectors)
    medoid_vectors = medoid_vectors / (np.linalg.norm(medoid_vectors, axis=1, keepdims=True) + 1e-10)

    cluster_id_to_medoid = {m['cluster']: m for m in medoids}
    cluster_id_to_vector = {m['cluster']: v for m, v in zip(medoids, medoid_vectors)}
    traj_id_to_vector = {t['traj_id']: np.array(t['pattern']).flatten() for t in trajectory_records}

    training_set = []
    reference_sets = {}

    # Compute cluster-to-cluster distance matrix
    cluster_dist = pairwise_distances(medoid_vectors, metric='euclidean')
    cluster_dist_argsort = np.argsort(cluster_dist, axis=1)  # pre-sorted for fast lookup

    for idx, cluster_id in enumerate(all_clusters):
        cluster_trajs = cluster_groups[cluster_id]
        medoid = cluster_id_to_medoid[cluster_id]
        train_tid = medoid['traj_id']
        training_set.append(medoid)

        in_cluster_refs = [t for t in cluster_trajs if t['traj_id'] != train_tid]
        needed = max(0, min_cluster_size - len(in_cluster_refs))
        collected_refs = in_cluster_refs.copy()
        seen_tids = {t['traj_id'] for t in collected_refs}
        seen_tids.add(train_tid)

        # Start checking other clusters in order of proximity
        neighbor_idx_list = cluster_dist_argsort[idx]
        for neighbor_idx in neighbor_idx_list:
            neighbor_cluster = all_clusters[neighbor_idx]
            if neighbor_cluster == cluster_id:
                continue  # skip self

            neighbor_trajs = cluster_groups[neighbor_cluster]
            candidate_trajs = [t for t in neighbor_trajs if t['traj_id'] not in seen_tids]

            if len(candidate_trajs) <= needed:
                collected_refs.extend(candidate_trajs)
                seen_tids.update(t['traj_id'] for t in candidate_trajs)
                needed -= len(candidate_trajs)
            else:
                # Pick top-N closest to medoid
                medoid_vec = cluster_id_to_vector[cluster_id]
                cand_vecs = np.stack([traj_id_to_vector[t['traj_id']] for t in candidate_trajs])
                cand_vecs = cand_vecs / (np.linalg.norm(cand_vecs, axis=1, keepdims=True) + 1e-10)
                dists = pairwise_distances([medoid_vec], cand_vecs, metric='euclidean')[0]
                sorted_idxs = np.argsort(dists)[:needed]
                selected = [candidate_trajs[i] for i in sorted_idxs]
                collected_refs.extend(selected)
                break  # done

            if needed <= 0:
                break

        reference_sets[train_tid] = collected_refs

    return training_set, reference_sets

training_set, reference_sets =  build_reference_sets_cluster_enrichment(trajectory_records=trajectory_records, medoids=medoids, min_cluster_size=min_cluster_size)

In [48]:
training_set[0]

{'traj_id': '25070976_25071072',
 'home': (160.0, 83.0),
 'work': (119.0, 115.0),
 'DeviceID': '43708',
 'pattern': [119.0, 115.0, 52.009614495783374],
 'cluster': 0}

In [49]:
cluster_sizes= []
for key in reference_sets:
    cluster_sizes.append(len(reference_sets[key]))
pd.Series(cluster_sizes).describe()

count    2000.000000
mean      213.863000
std       151.341717
min       100.000000
25%       100.000000
50%       151.500000
75%       270.000000
max      1106.000000
dtype: float64

In [50]:
if current_selection == 'Home':
    save_dict(training_set, save_path_base_home / "training_set.pkl")
    save_dict(reference_sets, save_path_base_home / "reference_sets.pkl")
elif current_selection == 'Work':
    save_dict(training_set, save_path_base_work / "training_set.pkl")
    save_dict(reference_sets, save_path_base_work / "reference_sets.pkl")


Saved to: PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Work/training_set.pkl
Saved to: PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Work/reference_sets.pkl


In [51]:
def save_training_and_reference_sets(training_set, reference_sets, df, output_folder):
    """
    Save training and reference trajectory sets into CSV files with safety checks.

    Args:
        training_set: List of training trajectory dicts (with 'traj_id')
        reference_sets: Dict {train_traj_id → list of trajectory dicts}
        df: Original DataFrame with all GPS records (must include 'traj_id', 'Latitude', 'Longitude', 'Timestamp')
        output_folder: Folder path to save training_set.csv, cluster_*.csv, and mapping
    """
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    df = df.set_index('traj_id')
    training_rows = []
    mapping_rows = []

    print(f"\n📦 Saving training and reference sets to {output_path.resolve()}")

    for train_traj in training_set:
        train_tid = train_traj['traj_id']
        cluster_file = f"cluster_{train_tid}.csv"
        out_file = output_path / cluster_file

        # Sanity check: already saved?
        if out_file.exists():
            print(f"⏭️ Skipping {cluster_file}, already exists.")
            continue

        print(f"🧭 Processing {cluster_file}")
        mapping_rows.append({
            'cluster_file': cluster_file,
            'representant_tid': train_tid
        })

        # Save training trajectory
        try:
            tid_min, tid_max = map(int, train_tid.split('_'))
            train_df = df.loc[tid_min:tid_max].copy()
            if train_df.empty:
                print(f"⚠️  Skipped empty training trajectory {train_tid}")
                continue
            train_df['tid'] = train_tid
            training_rows.append(train_df)
        except KeyError:
            print(f"⚠️  Training trajectory {train_tid} not found in df.")
            continue

        # Build and save reference set
        ref_trajs = reference_sets.get(train_tid, [])
        ref_rows = []
        for ref in ref_trajs:
            ref_tid = ref['traj_id']
            try:
                tid_min, tid_max = map(int, ref_tid.split('_'))
                ref_df = df.loc[tid_min:tid_max].copy()
                if ref_df.empty:
                    continue
                ref_df['tid'] = ref_tid
                ref_rows.append(ref_df)
            except KeyError:
                continue

        if not ref_rows:
            print(f"⚠️  No reference trajectories found for {train_tid}")
            continue

        cluster_df = pd.concat(ref_rows).reset_index()
        if 'Latitude' not in cluster_df or 'Longitude' not in cluster_df:
            print(f"⚠️  Malformed reference set for {train_tid}")
            continue

        cluster_df = cluster_df[['tid', 'Timestamp', 'Latitude', 'Longitude']]
        cluster_df.rename(columns={'Timestamp': 'timestamp', 'Latitude': 'lat', 'Longitude': 'lon'}, inplace=True)
        cluster_df[['lat', 'lon']] = cluster_df.groupby('tid')[['lat', 'lon']].transform(lambda g: g.ffill())
        cluster_df.to_csv(out_file, index=False)

    # Save training set
    if training_rows:
        training_df = pd.concat(training_rows)
        training_df = training_df[['tid', 'Timestamp', 'Latitude', 'Longitude']]
        training_df.rename(columns={'Timestamp': 'timestamp', 'Latitude': 'lat', 'Longitude': 'lon'}, inplace=True)
        training_df[['lat', 'lon']] = training_df.groupby('tid')[['lat', 'lon']].transform(lambda g: g.ffill())
        training_df.to_csv(output_path / "training_set.csv", index=False)

        mapping_df = pd.DataFrame(mapping_rows)
        mapping_df.to_csv(output_path / "representant_mapping.txt", index=False)

        print(f"\n✅ Saved training set with {len(training_rows)} trajectories.")
        print(f"📁 Reference clusters saved to: {output_path.resolve()}")
    else:
        print("⚠️ No valid training samples saved.")

if current_selection == 'Home':
    save_training_and_reference_sets(training_set, reference_sets, df, save_path_data_datasets_home)
elif current_selection == 'Work':
    save_training_and_reference_sets(training_set, reference_sets, df, save_path_data_datasets_work)


📦 Saving training and reference sets to /home/akouamdj/mobleak-datasets/PreprocessedData/2SplittedData/YJMob100Kv3/NormalizationType2/Work/Datasets
🧭 Processing cluster_25070976_25071072.csv
🧭 Processing cluster_18740736_18740832.csv
🧭 Processing cluster_27638016_27638112.csv
🧭 Processing cluster_6935712_6935808.csv
🧭 Processing cluster_26271168_26271264.csv
🧭 Processing cluster_50416128_50416224.csv
🧭 Processing cluster_62065920_62066016.csv
🧭 Processing cluster_5349120_5349216.csv
🧭 Processing cluster_54887616_54887712.csv
🧭 Processing cluster_17947776_17947872.csv
🧭 Processing cluster_10143840_10143936.csv
🧭 Processing cluster_36286656_36286752.csv
🧭 Processing cluster_61293792_61293888.csv
🧭 Processing cluster_18660768_18660864.csv
🧭 Processing cluster_39455136_39455232.csv
🧭 Processing cluster_27574848_27574944.csv
🧭 Processing cluster_2193024_2193120.csv
🧭 Processing cluster_58122096_58122192.csv
🧭 Processing cluster_18836832_18836928.csv
🧭 Processing cluster_129024_129120.csv
🧭